# Exercise 2.3: 2D State-Space Model

This notebook extends Exercise 2.2 to a two-dimensional linear Gaussian state-space model for transformer temperature. The implementation is in Python, while the corresponding 2D R functions have been added to `functions_exercise2.R`.

## Model

The 2D model is

$$
X_{t+1} = A X_t + B u_t + \eta_t, \qquad \eta_t \sim N(0,Q),
$$

$$
Y_t = H X_t + \epsilon_t, \qquad \epsilon_t \sim N(0,R),
$$

where

$$
X_t \in \mathbb{R}^2, \qquad u_t = (T_{a,t}, S_t, I_t)^T, \qquad H = \begin{bmatrix}1 & 0\end{bmatrix}.
$$

The first state is directly connected to the observed transformer temperature. The second state is hidden and can capture additional thermal dynamics. Solar radiation is divided by 1000, so solar coefficients are interpreted per `1000 W/m^2`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize
from scipy.stats import probplot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.style.use("seaborn-v0_8-whitegrid")

if Path("ex2_3.ipynb").exists():
    notebook_dir = Path.cwd()
elif Path("assignment4/ex2").exists():
    notebook_dir = Path("assignment4/ex2")
else:
    notebook_dir = Path.cwd()

data_candidates = [
    notebook_dir.parent / "transformer_data.csv",
    Path("assignment4/transformer_data.csv"),
    Path("../transformer_data.csv"),
]

data_path = next(path for path in data_candidates if path.exists())
output_dir = notebook_dir / "exercise2_3_outputs"
output_dir.mkdir(exist_ok=True)

raw_df = pd.read_csv(data_path)
df = raw_df.copy()
df["S"] = df["S"] / 1000.0

df.head()

In [ ]:
summary = df[["Y", "Ta", "S", "I"]].describe().T
summary

## Kalman Filter and Likelihood

The parameter vector is

$$
(A_{11}, A_{12}, A_{21}, A_{22}, B_{11}, B_{12}, B_{13}, B_{21}, B_{22}, B_{23}, \log q_{11}, q_{21}, \log q_{22}, \log r, x_{01}, x_{02}).
$$

The state covariance is constructed as \(Q = Q_{lt} Q_{lt}^T\), so it is positive semidefinite, and \(R = \exp(\log r)^2\), so the observation variance is positive.

In [ ]:
def map_par_2d(par):
    par = np.asarray(par, dtype=float)
    A = par[0:4].reshape(2, 2)
    B = par[4:10].reshape(2, 3)
    Qlt = np.array([
        [np.exp(par[10]), 0.0],
        [par[11], np.exp(par[12])],
    ])
    Q = Qlt @ Qlt.T
    H = np.array([[1.0, 0.0]])
    R = np.array([[np.exp(par[13]) ** 2]])
    X0 = par[14:16].reshape(2, 1)
    return A, B, Q, H, R, X0


def kf_filter_dt_2d(par, data):
    A, B, Q, H, R, X0 = map_par_2d(par)
    y = data["Y"].to_numpy(dtype=float)
    u = data[["Ta", "S", "I"]].to_numpy(dtype=float)
    n_obs = len(data)

    x_est = X0.copy()
    P_est = np.eye(2) * 10.0
    loglik = 0.0

    x_pred_store = np.empty((n_obs, 2))
    x_filt_store = np.empty((n_obs, 2))
    y_pred_store = np.empty(n_obs)
    innovation_store = np.empty(n_obs)
    innovation_var_store = np.empty(n_obs)

    for t in range(n_obs):
        u_t = u[t].reshape(3, 1)
        y_t = np.array([[y[t]]])

        x_pred = A @ x_est + B @ u_t
        P_pred = A @ P_est @ A.T + Q

        y_pred = H @ x_pred
        S_t = H @ P_pred @ H.T + R
        innovation = y_t - y_pred
        innovation_var = float(S_t[0, 0])

        if innovation_var <= 0 or not np.isfinite(innovation_var):
            return None

        innovation_scalar = float(innovation[0, 0])
        loglik += -0.5 * (
            np.log(2 * np.pi)
            + np.log(innovation_var)
            + innovation_scalar**2 / innovation_var
        )

        if not np.isfinite(loglik):
            return None

        K_t = P_pred @ H.T / innovation_var
        x_est = x_pred + K_t * innovation_scalar
        P_est = P_pred - K_t @ H @ P_pred

        x_pred_store[t] = x_pred.ravel()
        x_filt_store[t] = x_est.ravel()
        y_pred_store[t] = float(y_pred[0, 0])
        innovation_store[t] = innovation_scalar
        innovation_var_store[t] = innovation_var

    std_resid = innovation_store / np.sqrt(innovation_var_store)

    return {
        "loglik": float(loglik),
        "x_pred": x_pred_store,
        "x_filt": x_filt_store,
        "y_pred": y_pred_store,
        "innovation": innovation_store,
        "innovation_var": innovation_var_store,
        "std_resid": std_resid,
        "A": A,
        "B": B,
        "Q": Q,
        "H": H,
        "R": R,
        "X0": X0,
    }


def kf_loglik_dt_2d(par, data):
    kf = kf_filter_dt_2d(par, data)
    if kf is None or not np.isfinite(kf["loglik"]):
        return -np.inf
    return kf["loglik"]


def estimate_dt_2d(start_par, data, lower, upper, maxiter=2000):
    def neg_loglik(par):
        ll = kf_loglik_dt_2d(par, data)
        if not np.isfinite(ll):
            return 1e12
        return -ll

    return minimize(
        neg_loglik,
        np.asarray(start_par, dtype=float),
        method="L-BFGS-B",
        bounds=list(zip(lower, upper)),
        options={"maxiter": maxiter, "maxfun": 100000, "ftol": 1e-10, "gtol": 1e-7},
    )

In [ ]:
Y = df["Y"].to_numpy(dtype=float)

lower_2d = np.array([
    -1.2, -1.2, -1.2, -1.2,
    -10, -10, -10, -10, -10, -10,
    np.log(1e-4), -10, np.log(1e-4), np.log(1e-4),
    Y.min() - 20, -50,
])

upper_2d = np.array([
    1.2, 1.2, 1.2, 1.2,
    10, 10, 10, 10, 10, 10,
    np.log(20), 10, np.log(20), np.log(20),
    Y.max() + 20, 50,
])

# Best solution from a multi-start L-BFGS-B fit. The optimization is relatively slow
# because the 2D model has 16 parameters, so the saved best parameters are used
# by default for reproducible notebook execution.
best_par_2d = np.array([
    0.823942868, 0.160722667, -0.465011984, -0.0182016499,
    -0.00908166148, 1.09983950, 0.0700513629,
    0.855283033, 7.56576550, 1.35537679,
    -0.966539054, 2.14105548, -5.38144515,
    -1.88994933, 26.6001076, 0.336374268,
])

RUN_OPTIMIZATION = False

if RUN_OPTIMIZATION:
    start_grid = [
        best_par_2d,
        np.array([0.80, 0.05, 0.02, 0.70, 0.10, 2.0, 0.20, 0.01, 0.5, 0.05, np.log(0.5), 0.0, np.log(0.5), np.log(0.2), Y[0], 0.0]),
        np.array([0.80, 0.00, 0.10, 0.90, 0.10, 2.8, 0.20, 0.00, 0.2, 0.00, np.log(0.6), 0.0, np.log(0.3), np.log(0.1), Y[0], 0.0]),
    ]
    fits = [estimate_dt_2d(start, df, lower_2d, upper_2d, maxiter=2000) for start in start_grid]
    fit_2d = min(fits, key=lambda result: result.fun)
    par_2d = fit_2d.x
    optimization_summary = pd.DataFrame({
        "start": range(1, len(fits) + 1),
        "success": [result.success for result in fits],
        "negative_loglik": [result.fun for result in fits],
        "iterations": [result.nit for result in fits],
        "message": [str(result.message) for result in fits],
    })
else:
    par_2d = best_par_2d
    fit_2d = None
    optimization_summary = pd.DataFrame({
        "start": ["saved best multi-start solution"],
        "success": [False],
        "negative_loglik": [-kf_loglik_dt_2d(par_2d, df)],
        "iterations": [np.nan],
        "message": ["Best multi-start solution used for reproducible execution; set RUN_OPTIMIZATION=True to refit."],
    })

kf_2d = kf_filter_dt_2d(par_2d, df)
optimization_summary

In [ ]:
A_hat = kf_2d["A"]
B_hat = kf_2d["B"]
Q_hat = kf_2d["Q"]
H_hat = kf_2d["H"]
R_hat = kf_2d["R"]
X0_hat = kf_2d["X0"]

A_table = pd.DataFrame(A_hat, index=["State 1", "State 2"], columns=["State 1", "State 2"])
B_table = pd.DataFrame(B_hat, index=["State 1", "State 2"], columns=["Ta", "S", "I"])
Q_table = pd.DataFrame(Q_hat, index=["State 1", "State 2"], columns=["State 1", "State 2"])
R_table = pd.DataFrame(R_hat, index=["Y"], columns=["Y"])
X0_table = pd.DataFrame(X0_hat, index=["State 1", "State 2"], columns=["X0"])

A_table.to_csv(output_dir / "exercise2_3_A_matrix.csv")
B_table.to_csv(output_dir / "exercise2_3_B_matrix.csv")
Q_table.to_csv(output_dir / "exercise2_3_Q_matrix.csv")
R_table.to_csv(output_dir / "exercise2_3_R_matrix.csv")
X0_table.to_csv(output_dir / "exercise2_3_X0.csv")

print("A matrix")
display(A_table)
print("B matrix")
display(B_table)
print("Q matrix")
display(Q_table)
print("R matrix")
display(R_table)
print("Initial state")
display(X0_table)

### Estimated Matrices

Estimated transition matrix \(A\):

|  | State 1 | State 2 |
|---|---:|---:|
| State 1 | 0.8239 | 0.1607 |
| State 2 | -0.4650 | -0.0182 |

Estimated input matrix \(B\):

|  | Ta | S | I |
|---|---:|---:|---:|
| State 1 | -0.0091 | 1.0998 | 0.0701 |
| State 2 | 0.8553 | 7.5658 | 1.3554 |

Estimated state covariance \(Q\):

|  | State 1 | State 2 |
|---|---:|---:|
| State 1 | 0.1447 | 0.8145 |
| State 2 | 0.8145 | 4.5841 |

Estimated observation covariance: \(R = 0.0228\). Estimated initial state: \(X_0 = (26.6001, 0.3364)^T\).

## Reconstructed States and Inputs

The next figures show the two filtered state trajectories and the input variables on the same time axis. These plots help interpret what each latent state may represent physically.

In [ ]:
filtered_states = pd.DataFrame({
    "time": time if "time" in globals() else df["time"].to_numpy(),
    "state_1": kf_2d["x_filt"][:, 0],
    "state_2": kf_2d["x_filt"][:, 1],
})
filtered_states.to_csv(output_dir / "exercise2_3_filtered_states.csv", index=False)

plot_time = df["time"].to_numpy()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_time, kf_2d["x_filt"][:, 0], label="Filtered state 1", color="tab:blue", linewidth=1.8)
ax.plot(plot_time, kf_2d["x_filt"][:, 1], label="Filtered state 2", color="tab:orange", linewidth=1.8)
ax.set_title("2D model: reconstructed filtered state trajectories")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("State value")
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig(output_dir / "exercise2_3_state_trajectories.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
axes[0].plot(plot_time, kf_2d["x_filt"][:, 0], label="State 1", color="tab:blue", linewidth=1.6)
axes[0].plot(plot_time, kf_2d["x_filt"][:, 1], label="State 2", color="tab:orange", linewidth=1.6)
axes[0].set_ylabel("States")
axes[0].set_title("Filtered states and input variables")
axes[0].legend(loc="upper right")

axes[1].plot(plot_time, df["Ta"], color="tab:green", linewidth=1.3)
axes[1].set_ylabel("Ta")

axes[2].plot(plot_time, df["S"], color="tab:red", linewidth=1.3)
axes[2].set_ylabel("S / 1000")

axes[3].plot(plot_time, df["I"], color="tab:purple", linewidth=1.3)
axes[3].set_ylabel("I")
axes[3].set_xlabel("Time [hours]")

for axis in axes:
    axis.grid(True, alpha=0.3)

fig.savefig(output_dir / "exercise2_3_states_inputs.png", dpi=160, bbox_inches="tight")
plt.show()

filtered_states.describe().T

![Filtered state trajectories](exercise2_3_outputs/exercise2_3_state_trajectories.png)

![Filtered states and input variables](exercise2_3_outputs/exercise2_3_states_inputs.png)

The first state closely follows the observed transformer temperature because the observation matrix is fixed at `H = [1, 0]`. The second state has a similar daily pattern but is not directly observed. Since it is strongly affected by solar radiation and load in the estimated `B` matrix, and then influences state 1 through `A12`, it can be interpreted as an additional thermal forcing or buffer state. This interpretation is physically plausible, but it should be treated cautiously because the state itself is latent.

In [ ]:
loglik_2d = kf_2d["loglik"]
k_2d = len(par_2d)
n = len(df)
AIC_2d = 2 * k_2d - 2 * loglik_2d
BIC_2d = np.log(n) * k_2d - 2 * loglik_2d
RMSE_2d = np.sqrt(np.mean((df["Y"].to_numpy() - kf_2d["y_pred"]) ** 2))

criteria_2d = pd.DataFrame({
    "metric": ["log-likelihood", "AIC", "BIC", "RMSE"],
    "value": [loglik_2d, AIC_2d, BIC_2d, RMSE_2d],
})
criteria_2d.to_csv(output_dir / "exercise2_3_model_criteria.csv", index=False)
criteria_2d

### 2D Model Criteria

| Metric | Value |
|---|---:|
| log-likelihood | -125.1668 |
| AIC | 282.3336 |
| BIC | 332.3170 |
| RMSE | 0.5056 |

In [ ]:
criteria_1d = pd.read_csv(notebook_dir / "exercise2_2_outputs" / "exercise2_2_model_criteria.csv")
lookup_1d = dict(zip(criteria_1d["metric"], criteria_1d["value"]))

comparison = pd.DataFrame({
    "model": ["1D", "2D"],
    "k": [7, k_2d],
    "log_likelihood": [lookup_1d["log-likelihood"], loglik_2d],
    "AIC": [lookup_1d["AIC"], AIC_2d],
    "BIC": [lookup_1d["BIC"], BIC_2d],
    "RMSE": [lookup_1d["RMSE"], RMSE_2d],
})
comparison.to_csv(output_dir / "exercise2_3_model_comparison.csv", index=False)
comparison

### Comparison With 1D Model

| Model | k | Log-likelihood | AIC | BIC | RMSE |
|---|---:|---:|---:|---:|---:|
| 1D | 7 | -165.4839 | 344.9678 | 366.8356 | 0.6426 |
| 2D | 16 | -125.1668 | 282.3336 | 332.3170 | 0.5056 |

## Prediction Plot

In [ ]:
time = df["time"].to_numpy()

def save_and_show(fig, filename):
    fig.savefig(output_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time, df["Y"], label="Observed Y", color="black", linewidth=1.8)
ax.plot(time, kf_2d["y_pred"], label="2D one-step-ahead prediction", color="tab:red", linestyle="--", linewidth=1.8)
ax.set_title("2D state-space model: observed vs one-step-ahead prediction")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Temperature")
ax.legend()
ax.grid(True, alpha=0.3)
save_and_show(fig, "exercise2_3_model_fit.png")

![Observed transformer temperature and 2D one-step-ahead prediction](exercise2_3_outputs/exercise2_3_model_fit.png)

## Residual Diagnostics

In [ ]:
std_resid_2d = kf_2d["std_resid"]
innovation_2d = kf_2d["innovation"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
axes = axes.ravel()

axes[0].plot(time, std_resid_2d, color="tab:green", linewidth=1.0)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Standardized residual series")
axes[0].set_xlabel("Time [hours]")
axes[0].set_ylabel("Standardized residual")
axes[0].grid(True, alpha=0.3)

plot_acf(std_resid_2d, lags=min(48, len(std_resid_2d) - 1), ax=axes[1])
axes[1].set_title("ACF of standardized residuals")

plot_pacf(std_resid_2d, lags=min(48, len(std_resid_2d) - 1), ax=axes[2], method="ywm")
axes[2].set_title("PACF of standardized residuals")

probplot(std_resid_2d, dist="norm", plot=axes[3])
axes[3].set_title("QQ plot of standardized residuals")

save_and_show(fig, "exercise2_3_residual_diagnostics.png")

![Residual diagnostics for the 2D state-space model](exercise2_3_outputs/exercise2_3_residual_diagnostics.png)

In [ ]:
ljung_box_2d = acorr_ljungbox(std_resid_2d, lags=[10, 24], return_df=True)
residual_summary = pd.DataFrame({
    "metric": [
        "mean standardized residual",
        "std standardized residual",
        "max absolute standardized residual",
        "Ljung-Box p-value, lag 10",
        "Ljung-Box p-value, lag 24",
    ],
    "value": [
        std_resid_2d.mean(),
        std_resid_2d.std(ddof=1),
        np.max(np.abs(std_resid_2d)),
        ljung_box_2d.loc[10, "lb_pvalue"],
        ljung_box_2d.loc[24, "lb_pvalue"],
    ],
})
residual_summary.to_csv(output_dir / "exercise2_3_residual_summary.csv", index=False)
residual_summary

### Residual Summary

| Metric | Value |
|---|---:|
| Mean standardized residual | -0.0135 |
| Standard deviation of standardized residuals | 1.0010 |
| Maximum absolute standardized residual | 4.6211 |
| Ljung-Box p-value, lag 10 | 0.1671 |
| Ljung-Box p-value, lag 24 | 0.2486 |

## Discussion

The 2D model improves substantially over the 1D model. The log-likelihood increases from `-165.48` to `-125.17`, while AIC decreases from `344.97` to `282.33` and BIC decreases from `366.84` to `332.32`. The RMSE also decreases from `0.6426` to `0.5056`. These results indicate that the second latent state is justified despite the larger number of parameters.

The residual diagnostics also improve. In the 1D model, the residual lag-1 autocorrelation was about `0.325`, whereas in the 2D model it is about `0.068`. The Ljung-Box p-values at lags 10 and 24 are no longer small, which suggests that the 2D model removes much of the remaining serial dependence in the residuals.

The first state is directly observed through `H = [1, 0]`, so it can be interpreted as the transformer temperature component. The second state is latent. Since it is strongly affected by the inputs and feeds into the first state through the off-diagonal element `A12`, it may represent an additional thermal forcing or buffer component. This interpretation should be treated cautiously because latent states are not directly measured.

The 2D model follows the observed temperature more closely than the 1D model, especially during periods of more rapid changes. Some extreme residuals remain, and the QQ plot should still be checked for tail deviations, but overall the 2D model gives a clearer representation of the transformer thermal dynamics.